In [ ]:
import os
import json
import sys
sys.path.append(os.path.abspath('../'))

from retrieval_models import *

with open('../data/data.json') as f:
    data = json.loads(f.read())

In [ ]:
queries = data['queries_stopped_stemmed']
documents = data['documents_stopped_stemmed']
qrels = data['qrels']

print("Number of queries:", len(queries))
print("Number of documents,", len(documents))

In [ ]:
def mean_per_key(scores: dict[str, dict[str, float]]) -> dict[str, float]:
    q_ids = [key for key in scores.keys()]
    key_lists = {k: [v] for k, v in scores[q_ids[0]].items()}
    for q_id in q_ids[1:]:
        q_scores = scores[q_id]
        for k, v in q_scores.items():
            key_lists[k].append(v)
    return {k: sum(v) / len(v) for k, v in key_lists.items()}

In [ ]:
tf_results = compute_tf(queries, documents)
tf_scores = tf_results.compute_metrics(qrels)
mean_per_key(tf_scores)

In [ ]:
bm25_results = compute_bm25(queries, documents)
bm25_scores = bm25_results.compute_metrics(qrels)
mean_per_key(bm25_scores)

In [ ]:
ql_results = compute_ql(queries, documents)
ql_scores = ql_results.compute_metrics(qrels)
mean_per_key(ql_scores)

In [ ]:
def examine_results(results: RetrievalModelScores, q_id: str, top_k: int):
    query_text = queries[q_id]
    print("=" * 20)
    print(q_id)
    print(" ".join(query_text))
    print("=" * 20)
    
    doc_scores = results.doc_scores[q_id]
    score_pairs = sorted([(d_id, doc.score) for d_id, doc in doc_scores.items()], key=lambda x: x[1], reverse=True)[:top_k]
    for d_id, score in score_pairs:
        print(d_id, score)
        print(" ".join(documents[d_id]))
        scored_doc = doc_scores[d_id]
        print("Word scores:")
        for word, score in scored_doc.word_scores.items():
            print('\t', word, score)
        if scored_doc.missing_word_scores:
            print("Missing word scores:")
            for word, score in scored_doc.missing_word_scores.items():
                print(word, score, end=" ")
        print()
        print()

In [ ]:
examine_results(tf_results, '1110199', 10)

In [ ]:
examine_results(bm25_results, '1110199', 10)

In [ ]:
examine_results(ql_results, '1110199', 10)

In [ ]:
# compute ranges of values
k_1_range = np.linspace(0.0, 2.0, 5).tolist() + [10.0]
b_range = np.linspace(0.0, 1.0, 6).tolist()
bm25_range = compute_bm25_range(queries, documents, k_1_range, b_range)
bm25_range[0.0][0.0].doc_scores

In [ ]:
lambda_range = np.linspace(0.0, 1.0, 6).tolist()
ql_range = compute_ql_range(queries, documents, lambda_range)